# Part 11 — Appendix: Llama Guard 3 row of Table 1

Computes the second row of Table 1 (Section 1) using **Llama-Guard-3-8B** on the
300-prompt JBB judge_comparison split. Lives in its own kernel because Guard-3
loaded adjacent to any other vLLM model on a 24 GB GPU triggers an engine-core
race (see PLAN.md KNOWN BUGS + memory `feedback_guard3_end_of_notebook.md`).

`10_summary.ipynb` joins this Part's `summary.csv` with Part 1's into the
final Table 1.

**Outputs**: `results_part11/summary.csv`, `results_part11/raw.json`.


In [ ]:
import sys
sys.path.insert(0, '.')

from jbb_common import (
    setup_vllm_env, patch_jbb, hf_login, load_jbb_dataset,
    judge_metrics, refusal_keyword_check,
    format_llama2_chat, format_llama3_chat,
    JBB_SYSTEM_PROMPT, REFUSAL_KEYWORDS,
)
setup_vllm_env()
patch_jbb()
hf_login()
goals, benign_goals, judge_df = load_jbb_dataset()
print(f"Loaded {len(goals)} harmful goals, {len(benign_goals)} benign goals, "
      f"{len(judge_df)} judge rows")


In [ ]:
import os, json, gc
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

judge_prompts   = judge_df["prompt"].tolist()
judge_responses = judge_df["target_response"].tolist()

human_majority = judge_df["human_majority"]
if human_majority.dtype == bool:
    ground_truth = human_majority.tolist()
elif human_majority.dtype == object:
    ground_truth = human_majority.str.lower().isin(
        ["unsafe", "yes", "jailbroken", "true"]
    ).tolist()
else:
    ground_truth = human_majority.tolist()
print(f"Ground truth: {sum(ground_truth)} unsafe / "
      f"{len(ground_truth) - sum(ground_truth)} safe")

LG3_MODEL = "meta-llama/Llama-Guard-3-8B"
lg3_tokenizer = AutoTokenizer.from_pretrained(LG3_MODEL)

# (prompt, response) -> 2-turn chat with add_generation_prompt=True so Guard-3
# emits its safe / unsafe verdict for the assistant turn.
lg3_prompts = [
    lg3_tokenizer.apply_chat_template(
        [{"role": "user",      "content": p},
         {"role": "assistant", "content": r}],
        tokenize=False, add_generation_prompt=True,
    )
    for p, r in zip(judge_prompts, judge_responses)
]

llm_lg3 = LLM(
    model=LG3_MODEL, dtype="float16",
    max_model_len=4096, gpu_memory_utilization=0.90,
    tensor_parallel_size=1, enforce_eager=True,
)
lg3_sp = SamplingParams(temperature=0.0, max_tokens=50)
lg3_outputs = llm_lg3.generate(lg3_prompts, lg3_sp)

lg3_preds = []
for output in lg3_outputs:
    text = output.outputs[0].text.strip().lower()
    first_line = text.split("\n")[0]
    lg3_preds.append(first_line == "unsafe")

del llm_lg3, lg3_tokenizer
gc.collect()
torch.cuda.empty_cache()

metrics_lg3 = judge_metrics(lg3_preds, ground_truth)
print("Llama Guard 3 metrics:")
for k, v in metrics_lg3.items():
    print(f"  {k}: {v}")


In [ ]:
os.makedirs("results_part11", exist_ok=True)

summary_df = pd.DataFrame([{
    "Classifier": "Llama Guard 3",
    "Agreement":  metrics_lg3["agreement"],
    "FPR":        metrics_lg3["fpr"],
    "FNR":        metrics_lg3["fnr"],
}])
summary_df.to_csv("results_part11/summary.csv", index=False)

with open("results_part11/raw.json", "w") as f:
    json.dump({
        "predictions":  [bool(p) for p in lg3_preds],
        "ground_truth": [bool(g) for g in ground_truth],
        "metrics":      metrics_lg3,
    }, f, indent=2)

print("\n### Table 1 (Part 11 row): Llama Guard 3 judge ###")
print(summary_df.to_string(index=False))
